In [1]:
# --- Cell 1: imports ---
import os
import tempfile
import traceback
from typing import List, Tuple, Any

import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

from captum.influence import TracInCPFast  # ensure `pip install captum`

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
try:
    import captum
    print("captum:", captum.__version__)
except Exception:
    print("captum version: (unknown)")

C:\Users\marti\XAI4LLMsMetaToolkit\xai-llm-router\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.9.1+cpu
cuda available: False
captum: 0.8.0


In [2]:
# --- Cell 2: demo data (same as your plugin) ---
DEMO_POS = [
    "This product is absolutely amazing and works perfectly.",
    "I love this, it exceeded all my expectations.",
    "Fantastic quality and fast delivery, very happy.",
    "Best purchase I've made this year, highly recommend.",
    "Great value for money, works exactly as described.",
    "Incredible experience from start to finish.",
    "Would definitely buy this again without hesitation.",
    "The quality is outstanding and delivery was fast.",
    "Exceeded expectations in every single way.",
    "Superb build quality and excellent customer support.",
]
DEMO_NEG = [
    "Terrible product, broke after one day.",
    "Complete waste of money, very disappointed.",
    "Does not work as advertised, poor quality.",
    "Would not recommend, cheaply made and useless.",
    "Worst purchase ever, returning immediately.",
    "Absolutely dreadful — nothing works as described.",
    "Poor craftsmanship and zero customer support.",
    "Fell apart on first use, utter disappointment.",
    "Misleading description, product is a complete failure.",
    "Cheap materials, stopped working within a week.",
]

train_texts  = DEMO_POS + DEMO_NEG
train_labels = [1]*len(DEMO_POS) + [0]*len(DEMO_NEG)

test_text = "This item feels solid and arrived quickly, I am impressed."

print("n_train:", len(train_texts))

n_train: 20


In [3]:
# --- Cell 3: utilities + dataset ---
def dbg(x, name="x"):
    if torch.is_tensor(x):
        return f"{name}: shape={tuple(x.shape)} dtype={x.dtype} device={x.device}"
    return f"{name}: {type(x)} -> {x}"

class TextDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_len: int):
        enc = tokenizer(
            texts,
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        self.input_ids = enc["input_ids"]           # (N, L)
        self.attn_mask = enc["attention_mask"]      # (N, L)
        self.labels    = torch.tensor(labels, dtype=torch.long)  # (N,)
        self.texts     = texts

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # keep robust indexing
        if isinstance(idx, np.integer):
            idx = int(idx)
        if torch.is_tensor(idx):
            if idx.dim() == 0 or idx.numel() == 1:
                idx = int(idx.item())
            else:
                idx_list = idx.reshape(-1).tolist()
                return self.input_ids[idx_list], self.attn_mask[idx_list], self.labels[idx_list]
        return self.input_ids[idx], self.attn_mask[idx], self.labels[idx]

In [4]:
# --- Cell 4: model wrapper + "final fc" locator ---
class BERTWrapper(nn.Module):
    def __init__(self, hf_model):
        super().__init__()
        self.model = hf_model
        self._fc   = self._find_fc(hf_model)

    @staticmethod
    def _find_fc(hf_model) -> nn.Linear:
        # Try common heads first
        for attr in ("classifier", "score", "cls"):
            layer = getattr(hf_model, attr, None)
            if isinstance(layer, nn.Linear):
                return layer
            if layer is not None:
                for sub_attr in ("out_proj", "dense"):
                    sub = getattr(layer, sub_attr, None)
                    if isinstance(sub, nn.Linear):
                        return sub

        # fallback: last linear (can be wrong, but ok for debugging)
        last_linear = None
        for m in hf_model.modules():
            if isinstance(m, nn.Linear):
                last_linear = m
        if last_linear is None:
            raise RuntimeError("Could not locate a final nn.Linear in the model.")
        return last_linear

    @property
    def classifier(self) -> nn.Linear:
        return self._fc

    def forward(self, input_ids, attention_mask):
        # HF expects (B, L)
        return self.model(input_ids=input_ids, attention_mask=attention_mask).logits

In [5]:
# --- Cell 5: fine-tune + checkpoints ---
def fine_tune_with_checkpoints(
    texts: List[str],
    labels: List[int],
    model_name: str,
    device: str,
    max_len: int = 128,
    batch_size: int = 8,
    epochs: int = 2,
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    hf_model  = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model     = BERTWrapper(hf_model).to(device)

    dataset = TextDataset(texts, labels, tokenizer, max_len=max_len)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    optimizer   = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    total_steps = len(loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=max(1, total_steps // 10), num_training_steps=total_steps
    )
    loss_fn = nn.CrossEntropyLoss()

    ckpt_paths = []
    tmpdir = tempfile.TemporaryDirectory()

    for epoch in range(epochs):
        model.train()
        for ids, mask, lbls in loader:
            ids, mask, lbls = ids.to(device), mask.to(device), lbls.to(device)
            optimizer.zero_grad()
            logits = model(ids, mask)
            loss = loss_fn(logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        path = os.path.join(tmpdir.name, f"ckpt_epoch_{epoch+1}.pt")
        torch.save(model.state_dict(), path)
        ckpt_paths.append(path)

    return model, tokenizer, dataset, ckpt_paths, tmpdir

In [11]:
# --- Cell 5: fine-tune + checkpoints (with LR saved) ---
def fine_tune_with_checkpoints(
    texts: List[str],
    labels: List[int],
    model_name: str,
    device: str,
    max_len: int = 128,
    batch_size: int = 8,
    epochs: int = 2,
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    hf_model  = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model     = BERTWrapper(hf_model).to(device)

    dataset = TextDataset(texts, labels, tokenizer, max_len=max_len)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    optimizer   = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    total_steps = len(loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, total_steps // 10),
        num_training_steps=total_steps
    )
    loss_fn = nn.CrossEntropyLoss()

    ckpt_paths = []
    tmpdir = tempfile.TemporaryDirectory()

    for epoch in range(epochs):
        model.train()
        for ids, mask, lbls in loader:
            ids, mask, lbls = ids.to(device), mask.to(device), lbls.to(device)
            optimizer.zero_grad()
            logits = model(ids, mask)
            loss = loss_fn(logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        # LR at checkpoint time (end of epoch)
        lr = float(optimizer.param_groups[0]["lr"])

        path = os.path.join(tmpdir.name, f"ckpt_epoch_{epoch+1}.pt")
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "lr": lr,
                "epoch": epoch + 1,
            },
            path,
        )
        ckpt_paths.append(path)

    return model, tokenizer, dataset, ckpt_paths, tmpdir

In [15]:
# --- Cell 6: run end-to-end with lots of prints ---
MODEL_NAME = "distilbert-base-uncased"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS = 2
TOP_K = 5

model, tokenizer, train_dataset, ckpt_paths, tmpdir = fine_tune_with_checkpoints(
    train_texts, train_labels,
    model_name=MODEL_NAME,
    device=DEVICE,
    max_len=MAX_LEN,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS
)

print("checkpoints:", ckpt_paths)
print("final_fc_layer:", type(model.classifier), getattr(model.classifier, "weight", None).shape if hasattr(model.classifier, "weight") else None)

# Inspect one dataset item + one batch
s0 = train_dataset[0]
print(dbg(s0[0], "train_ids[0]"))
print(dbg(s0[1], "train_mask[0]"))
print(dbg(s0[2], "train_lbl[0]"))

b = next(iter(DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)))
print(dbg(b[0], "batch_ids"))
print(dbg(b[1], "batch_mask"))
print(dbg(b[2], "batch_lbls"))

# Build test tuple (BATCHED!)
enc = tokenizer(
    test_text,
    max_length=MAX_LEN,
    padding="max_length",
    truncation=True,
    return_tensors="pt",
)
test_ids  = enc["input_ids"].to(DEVICE)        # (1, L)
test_mask = enc["attention_mask"].to(DEVICE)   # (1, L)

with torch.no_grad():
    logits = model(test_ids, test_mask)        # (1, 2)
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
pred_label = int(np.argmax(probs))
test_lbl = torch.tensor([pred_label], dtype=torch.long, device=DEVICE)  # (1,)

test_tuple = (test_ids, test_mask, test_lbl)
print(dbg(test_tuple[0], "test_ids"))
print(dbg(test_tuple[1], "test_mask"))
print(dbg(test_tuple[2], "test_lbl"))
print("pred_label:", pred_label, "probs:", probs)

# Captum checkpoint loader: IMPORTANT -> must NOT return None if Captum multiplies its return
# We'll return the model (or True) explicitly to be safe.
def load_ckpt(m: nn.Module, p: str):
    obj = torch.load(p, map_location="cpu")
    #if isinstance(obj, dict) and "model_state_dict" in obj:
    m.load_state_dict(obj["model_state_dict"])
    lr = float(obj.get("lr", 1.0))

    m.to(DEVICE)
    return lr

tracin = TracInCPFast(
    model=model,
    final_fc_layer=model.classifier,
    train_dataset=train_dataset,
    checkpoints=ckpt_paths,
    checkpoints_load_func=load_ckpt,
    loss_fn=nn.CrossEntropyLoss(reduction="sum"),
    batch_size=BATCH_SIZE,
)

print("TracInCPFast created.")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


checkpoints: ['C:\\Users\\marti\\AppData\\Local\\Temp\\tmp9mrpfif0\\ckpt_epoch_1.pt', 'C:\\Users\\marti\\AppData\\Local\\Temp\\tmp9mrpfif0\\ckpt_epoch_2.pt']
final_fc_layer: <class 'torch.nn.modules.linear.Linear'> torch.Size([2, 768])
train_ids[0]: shape=(128,) dtype=torch.int64 device=cpu
train_mask[0]: shape=(128,) dtype=torch.int64 device=cpu
train_lbl[0]: shape=() dtype=torch.int64 device=cpu
batch_ids: shape=(8, 128) dtype=torch.int64 device=cpu
batch_mask: shape=(8, 128) dtype=torch.int64 device=cpu
batch_lbls: shape=(8,) dtype=torch.int64 device=cpu
test_ids: shape=(1, 128) dtype=torch.int64 device=cpu
test_mask: shape=(1, 128) dtype=torch.int64 device=cpu
test_lbl: shape=(1,) dtype=torch.int64 device=cpu
pred_label: 0 probs: [0.50379634 0.49620363]
TracInCPFast created.


In [17]:
# --- Cell 7: influence call with full traceback ---
try:
    out = tracin.influence(inputs=test_tuple)
    print("influence output type:", type(out))
    if torch.is_tensor(out):
        print(dbg(out, "out"))
        scores_row = out[0].detach().cpu()
    elif hasattr(out, "influence_scores"):
        print(dbg(out.influence_scores, "out.influence_scores"))
        scores_row = out.influence_scores[0].detach().cpu()
    else:
        raise RuntimeError(f"Unexpected Captum output: {type(out)}")

    sorted_desc = torch.argsort(scores_row, descending=True)
    prop_idx = sorted_desc[:TOP_K].tolist()
    opp_idx  = sorted_desc[-TOP_K:].flip(0).tolist()

    print("\nTop proponents:")
    for rank, i in enumerate(prop_idx, 1):
        print(rank, "score=", float(scores_row[i]), "| y=", int(train_labels[i]), "|", train_texts[i])

    print("\nTop opponents:")
    for rank, i in enumerate(opp_idx, 1):
        print(rank, "score=", float(scores_row[i]), "| y=", int(train_labels[i]), "|", train_texts[i])

except Exception as e:
    print("ERROR:", repr(e))
    print(traceback.format_exc())

influence output type: <class 'torch.Tensor'>
out: shape=(1, 20) dtype=torch.float32 device=cpu

Top proponents:
1 score= 0.00013260799460113049 | y= 0 | Complete waste of money, very disappointed.
2 score= 0.00013162830146029592 | y= 0 | Absolutely dreadful — nothing works as described.
3 score= 0.00012948304356541485 | y= 0 | Fell apart on first use, utter disappointment.
4 score= 0.00012396316742524505 | y= 0 | Cheap materials, stopped working within a week.
5 score= 0.0001235836825799197 | y= 0 | Would not recommend, cheaply made and useless.

Top opponents:
1 score= -0.00013403443153947592 | y= 1 | Exceeded expectations in every single way.
2 score= -0.00012569870159495622 | y= 1 | Superb build quality and excellent customer support.
3 score= -9.702603711048141e-05 | y= 1 | Incredible experience from start to finish.
4 score= -9.54134447965771e-05 | y= 1 | Best purchase I've made this year, highly recommend.
5 score= -8.621480810688809e-05 | y= 1 | Would definitely buy this again 